In [1]:
!pip install python-telegram-bot
!pip install ultralytics
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 56.8 MB/s eta 0:00:00


In [4]:
import nest_asyncio
nest_asyncio.apply()

import logging
import os
import cv2
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import Application, MessageHandler, CommandHandler, filters, ContextTypes
from telegram.constants import ChatAction


from ultralytics import YOLO
import speech_recognition as sr
from pydub import AudioSegment



class botTelegram:
    def __init__(self, token):
        self.__token = token
        self.app = Application.builder().token(self.__token).build()
        self.configuracao_handlers()

#-----------------------------------------------------------------------------------
#   Fizemos a instalação das bibliotecas necessarias para iniciar o funcionamento
# do bot.

#   O bot inicialmente recebe o token e coloca ele como um atributo privado, ja que
# não é interressante que tenham acesso à essa dinâmica interna

#   Contruimos a "alimentação" do nosso Bot, aquele que "bombea combustível"
# para que toda uma estrutura possa funcionar
#-----------------------------------------------------------------------------------

    def tipo_mensagem(self, update):
        if update.message.text:
            return "texto"
        elif update.message.photo:
            return "imagem"
        elif update.message.audio or update.message.voice:
            return "áudio"
        else:
            return "inválido"

    async def envio_mensagem(self, update, texto, reply_markup=None):

        await update.message.reply_text(texto, parse_mode="Markdown", reply_markup=reply_markup)

    async def mostrar_acao(self, update, acao):

        await update.message.chat.send_action(action=acao)

    async def download_imagem(self, update, local):
        imagem = update.message.photo[-1]
        arquivo_imagem = await imagem.get_file()
        await arquivo_imagem.download_to_drive(custom_path=local)
        return local

#-----------------------------------------------------------------------------------
#   O update funciona como um compartimento para onde os dados enviados pelo usuario
# vão, então, para acessar o que o usuario enviou, usamos "update.message"

#   usamos o async para que o codigo não fique esperando essa função terminar de "rodar"
# para que ai sim possamos realizar outras tarefas no código.

#   Se enviamos uma imagem, pegamos o ultimo elemento da lista para garantir uma
# melhor resolução

#   usamos o await como uma pausa, esperar o download da imagem/ ou esperar o app do
# telegram confirmar o recebimento da imagem

#   Logo em seguida, baixamos esse arquivo e colocamos ele em um "local", como se
# fosse uma "pasta" para podermos acessar/ trabalhar com esses arquivos dentro do
# código
#-----------------------------------------------------------------------------------

    async def download_audio(self, update, local):
        if update.message.voice:
            audio = update.message.voice
        else:
            audio = update.message.audio
        arquivo_audio = await audio.get_file()
        await arquivo_audio.download_to_drive(custom_path=local)
        return local



    def configuracao_handlers(self):

        handler_start = CommandHandler("start", self.start)
        self.app.add_handler(handler_start)


        handler_texto = MessageHandler(filters.TEXT, self.processamento_mensagem)
        self.app.add_handler(handler_texto)


        handler_imagem = MessageHandler(filters.PHOTO, self.processamento_mensagem)
        self.app.add_handler(handler_imagem)

        handler_audio = MessageHandler(filters.VOICE | filters.AUDIO, self.processamento_mensagem)
        self.app.add_handler(handler_audio)

        handler_invalido = MessageHandler(filters.ALL, self.processamento_mensagem)
        self.app.add_handler(handler_invalido)

#-----------------------------------------------------------------------------------
#   Ao receber uma mensagem, o "MessageHandler" aplica um filtro que retornará
# TRUE, se for audio/imagem. Isso acontecendo, o método "processamento_mensagem" aturará
# no reconhecimento/ processamento da mensagem.
#-----------------------------------------------------------------------------------

    async def start(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        await self.mostrar_acao(update, ChatAction.TYPING)

        boas_vindas = (
            "👋 *Olá! O bot foi iniciado.*\n\n"
            "Eu sou um assistente equipado com Inteligência Artificial! Consigo fazer duas coisas principais:\n"
            "1️⃣ *Analisar fotos* e detectar objetos usando o modelo YOLOv8.\n"
            "2️⃣ *Transcrever e classificar áudios* ou mensagens de voz.\n\n"
            "    Envie uma mensagem.\n\n"

        )
        await self.envio_mensagem(update, boas_vindas)

    async def processamento_mensagem(self, update, context: ContextTypes.DEFAULT_TYPE):

        tipo = self.tipo_mensagem(update)

        if tipo == "texto":
            texto_usuario = update.message.text
            await self.mostrar_acao(update, ChatAction.TYPING)
            await self.envio_mensagem(update,f"Você enviou uma mensagem de texto:\n\n{texto_usuario}")

        elif tipo == "imagem":
            await self.mostrar_acao(update, ChatAction.TYPING)
            await self.envio_mensagem(update, "📥 *Recebi sua imagem!* Processando com YOLOv8, aguarde um instante...")

            caminho_imagem = "imagem_recebida.jpg"
            await self.download_imagem(update, caminho_imagem)

            texto_resposta, foto_para_enviar = bot_img.processar_midia(caminho_imagem)

            await self.envio_mensagem(update, texto_resposta)

            if foto_para_enviar is not None:
                await self.mostrar_acao(update, ChatAction.UPLOAD_PHOTO)
                with open(foto_para_enviar, 'rb') as arquivo_foto:
                    await update.message.reply_photo(arquivo_foto)

        elif tipo == "áudio":
            await self.mostrar_acao(update, ChatAction.TYPING)
            await self.envio_mensagem(update, "📥 *Recebi seu áudio!* Convertendo e transcrevendo, aguarde...")

            caminho_audio_ogg = "audio_recebido.ogg"
            await self.download_audio(update, caminho_audio_ogg)

            resposta = bot_audio.processar_midia(caminho_audio_ogg)
            await self.envio_mensagem(update, resposta)

        else:
            await self.mostrar_acao(update, ChatAction.TYPING)
            await self.envio_mensagem(update, "Formato não suportado. Eu não consigo processar esse tipo de mensagem. Por favor, envie apenas **textos**, **fotos** ou **audios**")

#-----------------------------------------------------------------------------------
#   Agora, criamos um método que servirá como uma estrutura de polimorfismo para
# as classes filhas, de modo que as classes filhas poderão usar a dinâmica desse
# método de acordo com as necessidades de suas respectivas funções dentro do código
#-----------------------------------------------------------------------------------

    def iniciar(self):
        print("Olá! O bot foi iniciado. Envie uma mensagem!")
        self.app.run_polling(close_loop=False)



class BotImagem(botTelegram):
    def __init__(self):
        print("Carregando modelo YOLOv8m...")
        self.modelo_yolo = YOLO("yolov8m.pt")

        classes_pt = [
            "pessoa", "bicicleta", "carro", "motocicleta", "avião", "ônibus", "trem", "caminhão", "barco",
            "semáforo", "hidrante", "placa de pare", "parquímetro", "banco", "pássaro", "gato", "cachorro",
            "cavalo", "ovelha", "vaca", "elefante", "urso", "zebra", "girafa", "mochila", "guarda-chuva",
            "bolsa", "gravata", "mala", "frisbee", "esquis", "snowboard", "bola esportiva", "pipa",
            "taco de beisebol", "luva de beisebol", "skate", "prancha de surfe", "raquete de tênis",
            "garrafa", "taça de vinho", "copo", "garfo", "faca", "colher", "tigela", "banana", "maçã",
            "sanduíche", "laranja", "brócolis", "cenoura", "cachorro-quente", "pizza", "donut", "bolo",
            "cadeira", "sofá", "vaso de planta", "cama", "mesa de jantar", "vaso sanitário", "televisão",
            "notebook", "mouse", "controle remoto", "teclado", "celular", "micro-ondas", "forno",
            "torradeira", "pia", "geladeira", "livro", "relógio", "vaso", "tesoura", "urso de pelúcia",
            "secador de cabelo", "escova de dentes"
        ]

        # Fazemos aqui a tradução do reconhecimento para português
        for i, nome in enumerate(classes_pt):
            self.modelo_yolo.names[i] = nome

    def processar_midia(self, caminho_imagem):
        try:
            resultados = self.modelo_yolo(caminho_imagem)

            # caixas de imagem
            imagem_anotada = resultados[0].plot()
            caminho_imagem_salva = "imagem_com_caixas.jpg"
            cv2.imwrite(caminho_imagem_salva, imagem_anotada)

            objetos_detectados = []
            for resultado in resultados:
                for caixa in resultado.boxes:
                    numero_da_classe = int(caixa.cls[0])
                    nome_da_classe = self.modelo_yolo.names[numero_da_classe]
                    objetos_detectados.append(nome_da_classe)

            if len(objetos_detectados) > 0:
                contagem = {}
                for objeto in objetos_detectados:
                    if objeto in contagem:
                        contagem[objeto] += 1
                    else:
                        contagem[objeto] = 1

                # Formatação visual amigável usando Markdown listas
                texto_resposta = "📊 *Análise Concluída!*\n\nIdentifiquei os seguintes elementos na imagem:\n"
                for objeto, quantidade in contagem.items():
                    texto_resposta += f"• *{quantidade}x* _{objeto}_\n"

                return texto_resposta, caminho_imagem_salva
            else:
                return "🔍 Não consegui identificar nenhum objeto conhecido nesta imagem. Tente uma foto com mais clareza!", None

        except Exception as erro:
            return f"❌ Erro ao processar a imagem: {erro}", None


class BotAudio(botTelegram):
    def __init__(self):
        self.reconhecedor = sr.Recognizer()

    def processar_midia(self, caminho_audio):
        try:
            # 1. Converte o .ogg do Telegram para .wav
            caminho_wav = caminho_audio.replace(".ogg", ".wav")
            audio_convertido = AudioSegment.from_ogg(caminho_audio)
            audio_convertido.export(caminho_wav, format="wav")

            # 2. Leitura do Áudio
            with sr.AudioFile(caminho_wav) as fonte:
                self.reconhecedor.adjust_for_ambient_noise(fonte, duration=0.5)
                dados_audio = self.reconhecedor.record(fonte)

            # 3. Transcrição
            texto_transcrito = self.reconhecedor.recognize_google(dados_audio, language="pt-BR")
            texto_minusculo = texto_transcrito.lower()

            # 4. Classificação de Intenções
            classificacao = "💬 *Conversa Genérica*"

            palavras_de_ajuda = ["ajuda", "socorro", "problema", "suporte"]
            for palavra in palavras_de_ajuda:
                if palavra in texto_minusculo:
                    classificacao = "🚨 *O usuário precisa de suporte / Alerta*"

            palavras_de_venda = ["comprar", "pedido", "orçamento", "preço", "quanto custa"]
            for palavra in palavras_de_venda:
                if palavra in texto_minusculo:
                    classificacao = "💰 *Intenção Comercial: Venda ou Orçamento*"

            # Retorno formatado de maneira organizada
            return f"📝 *Transcrição do Áudio:*\n_\"{texto_transcrito}\"_\n\n🎯 *Classificação da Intenção:*\n{classificacao}"

        except sr.UnknownValueError:
            return "🎙️ *Poxa, o áudio ficou um pouco confuso!* Não consegui decodificar as palavras. Pode tentar gravar novamente de forma um pouco mais clara?"
        except Exception as erro:
            return f"⚠️ *Ocorreu um erro interno ao processar o áudio:* {erro}"




if __name__ == "__main__":
    TOKEN_DO_SEU_BOT = "8567874007:AAFHIDTGnFGPTc2MaH4kzjNY6G5aqiQeZ0s"

    # Instanciamos as suas classes de IA para deixá-las prontas na memória
    bot_img = BotImagem()
    bot_audio = BotAudio()

    # Instanciamos o bot principal do Telegram e rodamos
    meu_bot = botTelegram(TOKEN_DO_SEU_BOT)
    meu_bot.iniciar()

Carregando modelo YOLOv8m...
Olá! O bot foi iniciado. Envie uma mensagem!

image 1/1 /content/imagem_recebida.jpg: 640x512 1 bottle, 2 wine glasss, 2 knifes, 3 bowls, 1 chair, 1 dining table, 1270.1ms
Speed: 4.1ms preprocess, 1270.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 512)
